## EVAL baseline — before/after

Notebook LOCAL (không GPU) đọc 2 file `eval_pred_*.json` (zero-shot + fine-tune v3) đã lưu từ Drive về `seminar/code/eval/`.
Tính lại từ JSON raw: **nhận biết** (Sens/Spec/Prec/F1) + **định vị** (recall@IoU, per-patient pIoU + CI) + **FP ca âm**, rồi in bảng **before/after**.

**Caveat:** ~25 bệnh nhân dương (CI rộng); chỉ 2 bệnh nhân hoàn toàn không-u nên "Specificity" yếu về thống kê + là phân biệt nội-bệnh-nhân, KHÔNG phải specificity lâm sàng; "kích thước/IoU" là proxy 2D. Đây là PoC.

In [ ]:
# === [0] EVAL BASELINE — Gemma4 zero-shot (before) vs fine-tune v3 (after) ===
# Chạy LOCAL, KHÔNG GPU. Chỉ cần 2 file eval_pred_*.json (đọc pred_boxes/gt_boxes/is_pos/pid; gens nếu có).
import os, json, glob
import numpy as np
from collections import defaultdict
from scipy.optimize import linear_sum_assignment

HERE = os.getcwd()                                   # chạy trong seminar/code/
EVAL_DIR = os.path.join(HERE, "eval")                # nơi để 2 file JSON
IOU_THR  = 0.25                                      # ngưỡng "trúng" (khớp cell metrics finetune)

def _find(tag):
    fs = sorted(glob.glob(os.path.join(EVAL_DIR, f"eval_pred_*{tag}*.json")), key=os.path.getmtime)
    return fs[-1] if fs else None

# Đặt path tường minh nếu muốn; None -> tự tìm file mới nhất theo tag
BEFORE_JSON = _find("zeroshot")                      # Gemma4 chưa train
AFTER_JSON  = _find("v3")                            # bản fine-tune tốt
print("BEFORE:", BEFORE_JSON)
print("AFTER :", AFTER_JSON)


In [ ]:
# === [1] Hàm tính metric (Hungarian IoU + detection nhị phân + FP) ===
def iou(b1, b2):
    if not b1 or not b2: return 0.0
    xa, ya = max(b1[0], b2[0]), max(b1[1], b2[1]); xb, yb = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    a1 = max(0, b1[2]-b1[0]) * max(0, b1[3]-b1[1]); a2 = max(0, b2[2]-b2[0]) * max(0, b2[3]-b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0.0

def match(preds, gts):
    if not preds or not gts: return [], len(preds), len(gts)
    M = np.zeros((len(preds), len(gts)))
    for i, p in enumerate(preds):
        for j, g in enumerate(gts): M[i, j] = iou(p, g)
    ri, ci = linear_sum_assignment(1 - M)
    return [float(M[r, c]) for r, c in zip(ri, ci)], len(preds), len(gts)

def boot_ci(vals, n=2000):
    vals = np.asarray(vals, float)
    if len(vals) < 2: return (float("nan"), float("nan"))
    rng = np.random.default_rng(0); idx = np.arange(len(vals))
    ms = [vals[rng.choice(idx, len(idx), replace=True)].mean() for _ in range(n)]
    return tuple(np.percentile(ms, [2.5, 97.5]))

def analyze(rows):
    pos = [r for r in rows if r["is_pos"]]; neg = [r for r in rows if not r["is_pos"]]
    # localization
    ng = sum(len(r["gt_boxes"]) for r in pos); tp = 0; pen = defaultdict(list)
    for r in pos:
        mi, npd, ngt = match(r["pred_boxes"], r["gt_boxes"])
        tp += sum(v > IOU_THR for v in mi); d = max(ngt, npd)
        pen[r["pid"]].append(sum(mi)/d if d else 0.0)
    pat_iou = np.array([np.mean(v) for v in pen.values()]) if pen else np.array([0.0])
    lo, hi = boot_ci(pat_iou)
    # detection nhị phân per-patient
    bp = defaultdict(lambda: [0, 0])
    for r in rows:
        bp[r["pid"]][0] |= int(bool(r["gt_boxes"])); bp[r["pid"]][1] |= int(len(r["pred_boxes"]) > 0)
    it = list(bp.values())
    TP = sum(g and p for g, p in it); FP = sum((not g) and p for g, p in it)
    TN = sum((not g) and not p for g, p in it); FN = sum(g and not p for g, p in it)
    sens = TP/(TP+FN) if TP+FN else 0.0; prec = TP/(TP+FP) if TP+FP else 0.0
    f1 = 2*prec*sens/(prec+sens) if prec+sens else 0.0
    spec = TN/(TN+FP) if TN+FP else 0.0
    # FP ca âm
    fp_sl = sum(len(r["pred_boxes"]) > 0 for r in neg)
    nbp = defaultdict(list)
    for r in neg: nbp[r["pid"]].append(len(r["pred_boxes"]) > 0)
    fp_pat = sum(any(v) for v in nbp.values())
    return dict(n_test=len(rows), n_pos_pat=len(pat_iou),
                det_sens=sens, det_spec=spec, det_prec=prec, det_f1=f1,
                loc_recall=tp/ng if ng else 0.0,
                loc_pIoU=float(pat_iou.mean()), ci_lo=lo, ci_hi=hi,
                fp_slice=fp_sl/len(neg) if neg else 0.0,
                fp_patient=fp_pat/len(nbp) if nbp else 0.0)
print("hàm iou/match/analyze OK")


In [ ]:
# === [2] Metric BASELINE (zero-shot) — chi tiết ===
assert BEFORE_JSON, "Chưa có JSON zero-shot -> chạy baseline_zeroshot.ipynb trước, tải file về eval/"
B = analyze(json.load(open(BEFORE_JSON, encoding="utf-8"))["test"])
print(f"== BASELINE Gemma4 ZERO-SHOT (before train) == test {B['n_test']} lát | {B['n_pos_pat']} bệnh nhân dương\n")
print(f"[NHẬN BIẾT có/không u — per-patient]")
print(f"  Sensitivity {B['det_sens']:.0%} | Specificity {B['det_spec']:.0%} | Precision {B['det_prec']:.0%} | F1 {B['det_f1']:.2f}")
print(f"[ĐỊNH VỊ]")
print(f"  recall@IoU>{IOU_THR} {B['loc_recall']:.0%} | per-patient pIoU {B['loc_pIoU']:.3f} CI95 [{B['ci_lo']:.3f}, {B['ci_hi']:.3f}]")
print(f"[CA ÂM] FP {B['fp_slice']:.1%} lát | {B['fp_patient']:.1%} bệnh nhân")
print("\n(Zero-shot thường không ra đúng format box -> pred rỗng -> mọi số ~0. Đây là baseline HỢP LỆ:")
print(" chứng minh fine-tune DẠY được task từ số 0.)")


In [ ]:
# === [3] Bảng BEFORE / AFTER (zero-shot vs fine-tune v3) ===
assert BEFORE_JSON and AFTER_JSON, "Cần CẢ 2 JSON (zeroshot + v3) trong eval/"
A = analyze(json.load(open(AFTER_JSON, encoding="utf-8"))["test"])
rows = [
    ("Detection F1 (bệnh nhân)",        "det_f1",     "{:.2f}"),
    ("  Sensitivity (bắt u)",           "det_sens",   "{:.0%}"),
    ("  Precision",                     "det_prec",   "{:.0%}"),
    ("Localization recall@.25 (lát)",   "loc_recall", "{:.0%}"),
    ("Localization pIoU (bệnh nhân)",   "loc_pIoU",   "{:.3f}"),
    ("False-positive (lát âm)",         "fp_slice",   "{:.1%}"),
]
print(f"{'Chỉ số':34}{'BEFORE (zero-shot)':>20}{'AFTER (v3)':>14}")
print("-" * 68)
for label, k, fmt in rows:
    print(f"{label:34}{fmt.format(B[k]):>20}{fmt.format(A[k]):>14}")
print(f"\n(test {A['n_test']} lát | {A['n_pos_pat']} bệnh nhân dương | IoU-thr {IOU_THR})")
print("=> Chênh lệch = đóng góp của fine-tune. Caveat: n nhỏ (~25 bn dương), CI rộng; 'Spec' dựa n=2 bn không-u.")
